# 5 — Evaluate / 模型评估

**Chapter 15 — File 5 of 5 / 第15章 — 第5个文件（共5个）**

---

## Summary / 总结

This script demonstrates **load doc into memory**.

本脚本演示 **load doc into memory**。

---
## Step 1 — Step 1

In [ ]:
import string
import re
from os import listdir
from numpy import array
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import load_model

---
## Step 2 — load doc into memory

In [ ]:
def load_doc(filename):

---
## Step 3 — open the file as read only

In [ ]:
file = open(filename, 'r')

---
## Step 4 — read all text

In [ ]:
text = file.read()

---
## Step 5 — close the file

In [ ]:
file.close()
	return text

---
## Step 6 — turn a doc into clean tokens

In [ ]:
def clean_doc(doc, vocab):

---
## Step 7 — split into tokens by white space

In [ ]:
tokens = doc.split()

---
## Step 8 — prepare regex for char filtering

In [ ]:
re_punc = re.compile('[%s]' % re.escape(string.punctuation))

---
## Step 9 — remove punctuation from each word

In [ ]:
tokens = [re_punc.sub('', w) for w in tokens]

---
## Step 10 — filter out tokens not in vocab

In [ ]:
tokens = [w for w in tokens if w in vocab]
	tokens = ' '.join(tokens)
	return tokens

---
## Step 11 — load all docs in a directory

In [ ]:
def process_docs(directory, vocab, is_train):
	documents = list()

---
## Step 12 — walk through all files in the folder

In [ ]:
for filename in listdir(directory):

---
## Step 13 — skip any reviews in the test set

In [ ]:
if is_train and filename.startswith('cv9'):
			continue
		if not is_train and not filename.startswith('cv9'):
			continue

---
## Step 14 — create the full path of the file to open

In [ ]:
path = directory + '/' + filename

---
## Step 15 — load the doc

In [ ]:
doc = load_doc(path)

---
## Step 16 — clean doc

In [ ]:
tokens = clean_doc(doc, vocab)

---
## Step 17 — add to list

In [ ]:
documents.append(tokens)
	return documents

---
## Step 18 — load and clean a dataset

In [ ]:
def load_clean_dataset(vocab, is_train):

---
## Step 19 — load documents

In [ ]:
neg = process_docs('txt_sentoken/neg', vocab, is_train)
	pos = process_docs('txt_sentoken/pos', vocab, is_train)
	docs = neg + pos

---
## Step 20 — prepare labels

In [ ]:
labels = array([0 for _ in range(len(neg))] + [1 for _ in range(len(pos))])
	return docs, labels

---
## Step 21 — fit a tokenizer

In [ ]:
def create_tokenizer(lines):
	tokenizer = Tokenizer()
	tokenizer.fit_on_texts(lines)
	return tokenizer

---
## Step 22 — integer encode and pad documents

In [ ]:
def encode_docs(tokenizer, max_length, docs):

---
## Step 23 — integer encode

In [ ]:
encoded = tokenizer.texts_to_sequences(docs)

---
## Step 24 — pad sequences

In [ ]:
padded = pad_sequences(encoded, maxlen=max_length, padding='post')
	return padded

---
## Step 25 — classify a review as negative or positive

In [ ]:
def predict_sentiment(review, vocab, tokenizer, max_length, model):

---
## Step 26 — clean review

In [ ]:
line = clean_doc(review, vocab)

---
## Step 27 — encode and pad review

In [ ]:
padded = encode_docs(tokenizer, max_length, [line])

---
## Step 28 — predict sentiment

In [ ]:
yhat = model.predict(padded, verbose=0)

---
## Step 29 — retrieve predicted percentage and label

In [ ]:
percent_pos = yhat[0,0]
	if round(percent_pos) == 0:
		return (1-percent_pos), 'NEGATIVE'
	return percent_pos, 'POSITIVE'

---
## Step 30 — load the vocabulary

In [ ]:
vocab_filename = 'vocab.txt'
vocab = load_doc(vocab_filename)
vocab = set(vocab.split())

---
## Step 31 — load all reviews

In [ ]:
train_docs, ytrain = load_clean_dataset(vocab, True)
test_docs, ytest = load_clean_dataset(vocab, False)

---
## Step 32 — create the tokenizer

In [ ]:
tokenizer = create_tokenizer(train_docs)

---
## Step 33 — define vocabulary size

In [ ]:
vocab_size = len(tokenizer.word_index) + 1
print('Vocabulary size: %d' % vocab_size)

---
## Step 34 — calculate the maximum sequence length

In [ ]:
max_length = max([len(s.split()) for s in train_docs])
print('Maximum length: %d' % max_length)

---
## Step 35 — encode data

In [ ]:
Xtrain = encode_docs(tokenizer, max_length, train_docs)
Xtest = encode_docs(tokenizer, max_length, test_docs)

---
## Step 36 — load the model

In [ ]:
model = load_model('model.h5')

---
## Step 37 — evaluate model on training dataset

In [ ]:
_, acc = model.evaluate(Xtrain, ytrain, verbose=0)
print('Train Accuracy: %.2f' % (acc*100))

---
## Step 38 — evaluate model on test dataset

In [ ]:
_, acc = model.evaluate(Xtest, ytest, verbose=0)
print('Test Accuracy: %.2f' % (acc*100))

---
## Step 39 — test positive text

In [ ]:
text = 'Everyone will enjoy this film. I love it, recommended!'
percent, sentiment = predict_sentiment(text, vocab, tokenizer, max_length, model)
print('Review: [%s]\nSentiment: %s (%.3f%%)' % (text, sentiment, percent*100))

---
## Step 40 — test negative text

In [ ]:
text = 'This is a bad movie. Do not watch it. It sucks.'
percent, sentiment = predict_sentiment(text, vocab, tokenizer, max_length, model)
print('Review: [%s]\nSentiment: %s (%.3f%%)' % (text, sentiment, percent*100))

---
## Learning Notes / 学习笔记

- **概念**: load doc into memory 是机器学习中的常用技术。  
  *load doc into memory is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Evaluate / 模型评估
# Complete Code / 完整代码
# ===============================

import string
import re
from os import listdir
from numpy import array
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import load_model

# load doc into memory
def load_doc(filename):
	# open the file as read only
	file = open(filename, 'r')
	# read all text
	text = file.read()
	# close the file
	file.close()
	return text

# turn a doc into clean tokens
def clean_doc(doc, vocab):
	# split into tokens by white space
	tokens = doc.split()
	# prepare regex for char filtering
	re_punc = re.compile('[%s]' % re.escape(string.punctuation))
	# remove punctuation from each word
	tokens = [re_punc.sub('', w) for w in tokens]
	# filter out tokens not in vocab
	tokens = [w for w in tokens if w in vocab]
	tokens = ' '.join(tokens)
	return tokens

# load all docs in a directory
def process_docs(directory, vocab, is_train):
	documents = list()
	# walk through all files in the folder
	for filename in listdir(directory):
		# skip any reviews in the test set
		if is_train and filename.startswith('cv9'):
			continue
		if not is_train and not filename.startswith('cv9'):
			continue
		# create the full path of the file to open
		path = directory + '/' + filename
		# load the doc
		doc = load_doc(path)
		# clean doc
		tokens = clean_doc(doc, vocab)
		# add to list
		documents.append(tokens)
	return documents

# load and clean a dataset
def load_clean_dataset(vocab, is_train):
	# load documents
	neg = process_docs('txt_sentoken/neg', vocab, is_train)
	pos = process_docs('txt_sentoken/pos', vocab, is_train)
	docs = neg + pos
	# prepare labels
	labels = array([0 for _ in range(len(neg))] + [1 for _ in range(len(pos))])
	return docs, labels

# fit a tokenizer
def create_tokenizer(lines):
	tokenizer = Tokenizer()
	tokenizer.fit_on_texts(lines)
	return tokenizer

# integer encode and pad documents
def encode_docs(tokenizer, max_length, docs):
	# integer encode
	encoded = tokenizer.texts_to_sequences(docs)
	# pad sequences
	padded = pad_sequences(encoded, maxlen=max_length, padding='post')
	return padded

# classify a review as negative or positive
def predict_sentiment(review, vocab, tokenizer, max_length, model):
	# clean review
	line = clean_doc(review, vocab)
	# encode and pad review
	padded = encode_docs(tokenizer, max_length, [line])
	# predict sentiment
	yhat = model.predict(padded, verbose=0)
	# retrieve predicted percentage and label
	percent_pos = yhat[0,0]
	if round(percent_pos) == 0:
		return (1-percent_pos), 'NEGATIVE'
	return percent_pos, 'POSITIVE'

# load the vocabulary
vocab_filename = 'vocab.txt'
vocab = load_doc(vocab_filename)
vocab = set(vocab.split())
# load all reviews
train_docs, ytrain = load_clean_dataset(vocab, True)
test_docs, ytest = load_clean_dataset(vocab, False)
# create the tokenizer
tokenizer = create_tokenizer(train_docs)
# define vocabulary size
vocab_size = len(tokenizer.word_index) + 1
print('Vocabulary size: %d' % vocab_size)
# calculate the maximum sequence length
max_length = max([len(s.split()) for s in train_docs])
print('Maximum length: %d' % max_length)
# encode data
Xtrain = encode_docs(tokenizer, max_length, train_docs)
Xtest = encode_docs(tokenizer, max_length, test_docs)
# load the model
model = load_model('model.h5')
# evaluate model on training dataset
_, acc = model.evaluate(Xtrain, ytrain, verbose=0)
print('Train Accuracy: %.2f' % (acc*100))
# evaluate model on test dataset
_, acc = model.evaluate(Xtest, ytest, verbose=0)
print('Test Accuracy: %.2f' % (acc*100))

# test positive text
text = 'Everyone will enjoy this film. I love it, recommended!'
percent, sentiment = predict_sentiment(text, vocab, tokenizer, max_length, model)
print('Review: [%s]\nSentiment: %s (%.3f%%)' % (text, sentiment, percent*100))
# test negative text
text = 'This is a bad movie. Do not watch it. It sucks.'
percent, sentiment = predict_sentiment(text, vocab, tokenizer, max_length, model)
print('Review: [%s]\nSentiment: %s (%.3f%%)' % (text, sentiment, percent*100))